In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def transform_excel_to_csv(input_file, output_file):
    print(f"Processing: {input_file}")
    
    # 1. 헤더 구조 파악용 상위 행 읽기
    try:
        df_raw = pd.read_excel(input_file, header=None, nrows=10)
    except Exception as e:
        print(f"파일 읽기 실패: {e}")
        return

    # 2. '약효분류코드'가 포함된 행 찾기
    header_row_idx = None
    for i in range(len(df_raw)):
        row_values = df_raw.iloc[i].values
        if any('약효분류코드' in str(val) for val in row_values):
            header_row_idx = i
            break
    
    if header_row_idx is None:
        print("오류: '약효분류코드'를 포함하는 행을 찾을 수 없습니다.")
        return
    
    print(f"  → 헤더 행 위치: {header_row_idx}번째 행")

    # 3. 상단 헤더(월 정보) 존재 여부 확인
    has_top_header = False
    if header_row_idx > 0:
        top_row_values = df_raw.iloc[header_row_idx - 1].values
        if any('년' in str(val) and '월' in str(val) for val in top_row_values):
            has_top_header = True

    # 4. 엑셀 다시 읽기
    if has_top_header:
        df = pd.read_excel(input_file, header=[header_row_idx - 1, header_row_idx])
    else:
        df = pd.read_excel(input_file, header=header_row_idx)

    # 5. 컬럼명 평탄화
    if isinstance(df.columns, pd.MultiIndex):
        new_columns = []
        for col in df.columns:
            clean_parts = [str(c).strip() for c in col if 'Unnamed' not in str(c) and str(c) != 'nan']
            new_columns.append('_'.join(clean_parts) if clean_parts else '')
        df.columns = new_columns
    else:
        df.columns = [str(c).strip() for c in df.columns]

    # 6. 컬럼명 표준화
    rename_dict = {}
    for col in df.columns:
        if '약효분류코드' in col:
            rename_dict[col] = '약효분류코드'
        elif '약효분류명' in col:
            rename_dict[col] = '약효분류명'
    df.rename(columns=rename_dict, inplace=True)

    if '약효분류코드' not in df.columns or '약효분류명' not in df.columns:
        print("오류: '약효분류코드' 또는 '약효분류명' 컬럼을 찾을 수 없습니다.")
        print("현재 컬럼 목록:", df.columns.tolist())
        return

    # 7. '계' 행 제거
    df = df[df['약효분류명'] != '계']

    # 8. Melt
    id_vars = ['약효분류코드', '약효분류명']
    value_vars = [col for col in df.columns if col not in id_vars]
    
    df_melted = df.melt(id_vars=id_vars, value_vars=value_vars, 
                        var_name='월_구분', value_name='값')

    # 9. 월_구분 분리
    split_cols = df_melted['월_구분'].str.split('_', n=1, expand=True)
    df_melted['진료년월_raw'] = split_cols[0]
    df_melted['구분'] = split_cols[1] if split_cols.shape[1] > 1 else np.nan
    df_melted['구분'] = df_melted['구분'].fillna('수량')

    # 10. 연도, 월 추출
    df_melted['연도'] = df_melted['진료년월_raw'].astype(str).str.extract(r'(\d{4})년')
    df_melted['월'] = df_melted['진료년월_raw'].astype(str).str.extract(r'(\d{1,2})월')
    df_melted = df_melted.dropna(subset=['월'])
    df_melted['월'] = df_melted['월'].astype(int)
    df_melted['연도'] = df_melted['연도'].astype(int)

    # ★★★ 수정: 진료년월을 'YYYY-MM' 형식으로 생성 ★★★
    df_melted['진료년월'] = (
        df_melted['연도'].astype(str).str.zfill(4) + '-' + 
        df_melted['월'].astype(str).str.zfill(2)
    )

    # '값' 컬럼을 숫자형으로 변환 ('-' 등은 NaN으로)
    df_melted['값'] = pd.to_numeric(df_melted['값'], errors='coerce')

    # 11. Pivot
    df_pivot = df_melted.pivot_table(
        index=['약효분류코드', '약효분류명', '진료년월', '연도', '월'],
        columns='구분',
        values='값',
        aggfunc='sum'
    ).reset_index()

    # 12. 컬럼명 변경
    df_pivot.rename(columns={
        '수량': '수량_천개', 
        '사용금액': '사용금액_백만원'
    }, inplace=True)

    if '수량_천개' not in df_pivot.columns: df_pivot['수량_천개'] = np.nan
    if '사용금액_백만원' not in df_pivot.columns: df_pivot['사용금액_백만원'] = np.nan

    final_columns = ['약효분류코드', '약효분류명', '진료년월', '연도', '월', '수량_천개', '사용금액_백만원']
    df_final = df_pivot[final_columns]

    # 13. 타입 정리 및 정렬
    df_final['약효분류코드'] = df_final['약효분류코드'].astype(str)
    df_final = df_final.sort_values(by=['약효분류코드', '연도', '월']).reset_index(drop=True)

    # 14. CSV 저장
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {output_file}")
    print(f"  → 총 {len(df_final)}행 저장됨")
    print(f"  → 진료년월 예시: {df_final['진료년월'].unique()[:5].tolist()}\n")

# --- 실행 ---

if __name__ == "__main__":
    file1 = "/workspaces/healthcareDA/data/201401_201612.xlsx"
    file2 = "/workspaces/healthcareDA/data/201701_201912.xlsx"
    
    output1 = "/workspaces/healthcareDA/data/201401_201612_transformed.csv"
    output2 = "/workspaces/healthcareDA/data/201701_201912_transformed.csv"
    
    transform_excel_to_csv(file1, output1)
    transform_excel_to_csv(file2, output2)

Processing: /workspaces/healthcareDA/data/201401_201612.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201401_201612_transformed.csv
  → 총 4500행 저장됨
  → 진료년월 예시: ['2014-01', '2014-02', '2014-03', '2014-04', '2014-05']

Processing: /workspaces/healthcareDA/data/201701_201912.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201701_201912_transformed.csv
  → 총 4464행 저장됨
  → 진료년월 예시: ['2017-01', '2017-02', '2017-03', '2017-04', '2017-05']

Processing: /workspaces/healthcareDA/data/201401_201612.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201401_201612_transformed.csv
  → 총 4500행 저장됨
  → 진료년월 예시: ['2014-01', '2014-02', '2014-03', '2014-04', '2014-05']

Processing: /workspaces/healthcareDA/data/201701_201912.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201701_201912_transformed.csv
  → 총 4464행 저장됨
  → 진료년월 예시: ['2017-01', '2017-02', '2017-03', '2017-04', '2017-05']



In [ ]:
file1 = "/workspaces/healthcareDA/data/201401_201612_transformed.csv"
file2 = "/workspaces/healthcareDA/data/201701_201912_transformed.csv"
file3 = "/workspaces/healthcareDA/data/meft_2020-2025.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df3 = df3.iloc[:,1:]

df = pd.concat([df1,df2,df3],  axis=0)
df.to_csv('meft_2014-2025.csv',index=False)
for idf in [df1,df2,df3]:
    display(idf.head())
    print(idf.dtypes)

,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,111.0,전신마취제,2014-01,2014,1,6176.0,2754.0
1,111.0,전신마취제,2014-02,2014,2,5747.0,2567.0
2,111.0,전신마취제,2014-03,2014,3,5813.0,2579.0
3,111.0,전신마취제,2014-04,2014,4,5587.0,2466.0
4,111.0,전신마취제,2014-05,2014,5,5370.0,2377.0


약효분류코드      float64
약효분류명           str
진료년월            str
연도            int64
월             int64
수량_천개       float64
사용금액_백만원    float64
dtype: object


,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,111.0,전신마취제,2017-01,2017,1,946.0,3210.0
1,111.0,전신마취제,2017-02,2017,2,870.0,2888.0
2,111.0,전신마취제,2017-03,2017,3,933.0,3086.0
3,111.0,전신마취제,2017-04,2017,4,843.0,2842.0
4,111.0,전신마취제,2017-05,2017,5,842.0,2999.0


약효분류코드      float64
약효분류명           str
진료년월            str
연도            int64
월             int64
수량_천개       float64
사용금액_백만원    float64
dtype: object


,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,110,중추신경계용약,2020-01,2020,1,NaN,NaN
1,111,전신마취제,2020-01,2020,1,1017.0,3849.0
2,112,최면진정제,2020-01,2020,1,23221.0,3355.0
3,113,항전간제,2020-01,2020,1,73785.0,19473.0
4,114,해열·진통·소염제,2020-01,2020,1,478458.0,74462.0


약효분류코드        int64
약효분류명           str
진료년월            str
연도            int64
월             int64
수량_천개       float64
사용금액_백만원    float64
dtype: object


In [16]:
import pandas as pd

df = pd.read_csv('/workspaces/healthcareDA/src/meft_2014-2025.csv')
df.head()

,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,111.0,전신마취제,2014-01,2014,1,6176.0,2754.0
1,111.0,전신마취제,2014-02,2014,2,5747.0,2567.0
2,111.0,전신마취제,2014-03,2014,3,5813.0,2579.0
3,111.0,전신마취제,2014-04,2014,4,5587.0,2466.0
4,111.0,전신마취제,2014-05,2014,5,5370.0,2377.0


In [2]:
df.to_csv('meft_2014-2025.csv',index=False)

## open api 

In [17]:
df['약효분류코드'] = df['약효분류코드'].astype(int).astype(str)
meft_code_array = df['약효분류코드'].unique()

df['진료년월'] = df['진료년월'].str.replace('-', '')
meft_yrm_array = df['진료년월'].unique()


meft_code_array,meft_yrm_array


(<ArrowStringArray>
 ['111', '112', '113', '114', '115', '116', '117', '119', '121', '122',
  ...
  '821', '999', '129', '393', '110', '319', '140', '230', '250', '310']
 Length: 133, dtype: str,
 <ArrowStringArray>
 ['201401', '201402', '201403', '201404', '201405', '201406', '201407',
  '201408', '201409', '201410',
  ...
  '202503', '202504', '202505', '202506', '202507', '202508', '202509',
  '202510', '202511', '202512']
 Length: 144, dtype: str)

In [18]:
# 133*144 # 19152
meft_yrm_array[:72] #2014~2019
133*72

9576

In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time

# 설정


def fetch_data_from_api(diag_ym, meft_div_no, insup_tp="0", cpmd_prsc_tp="01", num_of_rows=10000, page_no=1):
    SERVICE_KEY = "febd95a18f16ff444bf3af3074dc32b0e309532a828e8a509bea11c6cd4eb764"
    BASE_URL = "https://apis.data.go.kr/B551182/msupUserInfoService1.2/getMeftDivSickList1.2"
    
    params = {
        "ServiceKey": SERVICE_KEY,
        "numOfRows": num_of_rows,
        "pageNo": page_no,
        "diagYm": diag_ym,
        "insupTp": insup_tp, # 전체:0 
        "cpmdPrscTp": cpmd_prsc_tp, #조제 01, 처방 02
        "meftDivNo": meft_div_no
    }
    
    response = requests.get(BASE_URL, params=params, timeout=60)
    
    if response.status_code == 200:
        print("HTTP status:", response.status_code)
        # print(response.url)
        return response.text
    else:
        print("HTTP status:", response.status_code)
        print("Error fetching data from API.")
        return None

def parse_xml_to_dataframe(xml_data):
    root = ET.fromstring(xml_data)
    
    data_list = []
    for item in root.findall(".//item"):
        row = {
            "진료년월": item.findtext("diagYm"),
            "보험구분코드": item.findtext("insupTpCd"),
            "약효분류번호": item.findtext("meftDivNo"),
            "약효분류명": item.findtext("meftDivNoNm"),
            "상병코드": item.findtext("st3SickSym"),
            "상병명": item.findtext("st3SickSymNm"),
            "의약품사용금액": item.findtext("msupUseAmt"),
            "총사용량": item.findtext("totUseQty"),
        }
        data_list.append(row)
    
    df = pd.DataFrame(data_list)
    
    # 데이터 타입 변환
    df["의약품사용금액"] = pd.to_numeric(df["의약품사용금액"], errors="coerce")
    df["총사용량"] = pd.to_numeric(df["총사용량"], errors="coerce")
    
    return df

def save_dataframe_to_pq(df, diag_ym):
    filename = f"data_{diag_ym}.pq"
    df.to_parquet(filename, index=False)
    print(f"Data saved to {filename}")

for diag_ym in meft_yrm_array[::-1]: #[18:36] , [36:72] # 나머지는 로컬로 돌려라. codespace 120시간 제한있다.
    df_list_by_diag_ym = []
    for meft_div_no in meft_code_array:
        print(f"Fetching data for diagYm: {diag_ym}, meftDivNo: {meft_div_no}")
        xml_data = fetch_data_from_api(diag_ym, meft_div_no)
        
        if xml_data:
            try:
                root = ET.fromstring(xml_data)
                total_count = int(root.findtext(".//totalCount"))
                if total_count == 0:
                    print(f"No data available for diagYm: {diag_ym}, meftDivNo: {meft_div_no}")
                    continue
                print(f"Total count for diagYm: {diag_ym}, meftDivNo: {meft_div_no} is {total_count}")
            except ET.ParseError as e:
                print(f"XML parsing error for diagYm: {diag_ym}, meftDivNo: {meft_div_no}: {e}")
                continue    
            xml_df = parse_xml_to_dataframe(xml_data)
            df_list_by_diag_ym.append(xml_df)
            
            if xml_df.shape[0] > 9999:
                xml_data = fetch_data_from_api(diag_ym, meft_div_no,page_no=2)
                df2 = parse_xml_to_dataframe(xml_data)
                df_list_by_diag_ym.append(df2)
                
        else:
            print(f"No data fetched for diagYm: {diag_ym}, meftDivNo: {meft_div_no}")
        
        time.sleep(1)  # 1초 대기
        
    concat_df = pd.concat(df_list_by_diag_ym, axis=0)    
    save_dataframe_to_pq(concat_df, diag_ym) # 하나당 3분... 71개는... 213분? 35개는..105분..17개는...51분
    





In [ ]:
import glob
import pandas as pd

# df.to_parquet('meft_2014-2025.parquet')

list_of_parquet_files = glob.glob('data_*.pq')
list_of_parquet_files.sort()  # 파일명을 기준으로 정렬
df_list = []

# csvdf = pd.read_csv('data_201401.csv')
# try : 
#     csvdf['진료년월'] = csvdf['진료년월'].astype(str)
#     csvdf['보험구분코드'] = csvdf['보험구분코드'].astype(str)
#     csvdf['약효분류번호'] = csvdf['약효분류번호'].astype(str)
# except KeyError:
#     print("KeyError: 'diagYm' 컬럼이 존재하지 않습니다. CSV 파일의 컬럼명을 확인하세요. ")
    
# df_list = [csvdf]
# # print(df_list[0].shape)
# # display(df_list[0].dtypes)

for file in list_of_parquet_files:
    # if '2014' not in file:
    #     continue  # 이미 읽은 파일은 건너뜀
    print(f"Found parquet file: {file}")
    pq_df = pd.read_parquet(file)
    # display(pq_df.head())
    df_list.append(pq_df)
    print(pq_df.shape)
    # print(pq_df.dtypes)
    
df= pd.concat(df_list, axis=0)


In [7]:
import pandas as pd

def simple_profile(df):
    profile = pd.DataFrame({
        'Data Type': df.dtypes,
        'Total Rows': len(df),
        'Unique Count': df.nunique(),
        'Null Count': df.isnull().sum(),
        'Null Pct (%)': (df.isnull().sum() / len(df) * 100).round(2),
        'Sample Unique Values': [list(df[col].dropna().unique()[:12]) for col in df.columns]
    })
    return profile


# 사용 예시
profile_df = simple_profile(df)
print(profile_df)

df['진료년월'].unique()

        Data Type  Total Rows  Unique Count  Null Count  Null Pct (%)  \
진료년월          str    10262367            60           0          0.00   
보험구분코드        str    10262367             3           0          0.00   
약효분류번호        str    10262367           101           0          0.00   
약효분류명         str    10262367           101           0          0.00   
상병코드          str    10262367          2360           0          0.00   
상병명           str    10262367          2054      126675          1.23   
의약품사용금액     int64    10262367       2285268           0          0.00   
총사용량        int64    10262367        222095           0          0.00   

                                      Sample Unique Values  
진료년월     [201501, 201502, 201503, 201504, 201505, 20150...  
보험구분코드                                           [4, 5, 7]  
약효분류번호   [111, 112, 113, 114, 115, 117, 119, 121, 122, ...  
약효분류명    [전신마취제, 최면진정제, 항전간제, 해열·진통·소염제, 각성제,흥분제, 정신신경용...  
상병코드     [AA04, AA06, AA08, AA09, AA1

<ArrowStringArray>
['201501', '201502', '201503', '201504', '201505', '201506', '201507',
 '201508', '201509', '201510', '201511', '201512', '201601', '201602',
 '201603', '201604', '201605', '201606', '201607', '201608', '201609',
 '201610', '201611', '201612', '201701', '201702', '201703', '201704',
 '201705', '201706', '201707', '201708', '201709', '201710', '201711',
 '201712', '201801', '201802', '201803', '201804', '201805', '201806',
 '201807', '201808', '201809', '201810', '201811', '201812', '201901',
 '201902', '201903', '201904', '201905', '201906', '201907', '201908',
 '201909', '201910', '201911', '201912']
Length: 60, dtype: str

In [ ]:
df.to_parquet('', index=False)

sick 데이터 중복 점검
meft 데이터 - sick 데이터 합산 비교
성분 다운로드 데이터 ..왜하지? --> 성병/연령별/원내외 환자수 cumsum하기

## sick data 중복점검 + meft data와 금액,수량 합산비교

In [1]:
import glob
import pandas as pd

# data 폴더 내의 모든 pq 파일 검색
pq_files = sorted(glob.glob('data/*.pq'))

print("=== sick data (data/*.pq) 중복 점검 ===")
total_duplicate_rows = 0

for file_path in pq_files:
    # 전체 데이터를 불러오기 전 메모리 절약을 위해 정밀 체크
    df_temp = pd.read_parquet(file_path)
    dup_count = df_temp.duplicated().sum()
    total_duplicate_rows += dup_count
    
    print(f"[{file_path}] 총 행 수: {len(df_temp):,d} / 중복 행 수: {dup_count:,d} ({dup_count/len(df_temp)*100:.2f}%)")

print(f"\n전체 파일 총 중복 행 수: {total_duplicate_rows:,d}")

=== sick data (data/*.pq) 중복 점검 ===

전체 파일 총 중복 행 수: 0


In [38]:
import glob
import os
import pandas as pd

# 1. 파일 경로 검색 (상위/하위 경로 고려)
sick_files = glob.glob('../data/data_2015*.pq')

if not sick_files:
    print("❌ 'data/' 디렉토리에서 2015년 pq 파일을 찾을 수 없습니다.")
    print("현재 'data/' 폴더 내 파일 목록:")
    if os.path.exists('data'):
        print(os.listdir('data')[:10])  # data 폴더 내 파일 일부 출력
    else:
        print("'data' 디렉토리가 존재하지 않습니다.")
else:
    print(f"✅ 발견된 2015년 파일 수: {len(sick_files)}개")
    sick_list = []
    sick_cols = ['진료년월', '의약품사용금액', '총사용량']

    for file_path in sick_files:
        df_sick = pd.read_parquet(file_path, columns=sick_cols)
        # display(df_sick.head())
        grp = df_sick.groupby('진료년월').agg({
            '의약품사용금액': 'sum',
            '총사용량': 'sum'
        }).reset_index()
        sick_list.append(grp)

    
    # sick 데이터 병합
    sick_summary = pd.concat(sick_list, ignore_index=True)
    sick_summary = sick_summary.groupby('진료년월', as_index=False).sum()

    # 단위 변환 (원 -> 백만원, 개 -> 천개)
    sick_summary = sick_summary[['진료년월', '의약품사용금액', '총사용량']]
    sick_summary['sick_금액_백만원'] = sick_summary['의약품사용금액'] / 1_000_000
    sick_summary['sick_수량_천개'] = sick_summary['총사용량'] / 1_000
    sick_summary = sick_summary[['진료년월', 'sick_금액_백만원', 'sick_수량_천개']]

    display(sick_summary.dtypes)

    # 2. meft data 로드 및 2015년 필터링
    meft_df = pd.read_csv('meft_2014-2025.csv')
    meft_df['진료년월'] = meft_df['진료년월'].astype(str).str.replace('-', '')
    meft_2015 = meft_df[meft_df['진료년월'].str.startswith('2015')].copy()
    print(f"✅ meft_2015 데이터 행 수: {len(meft_2015)}")
    meft_summary = meft_2015.groupby('진료년월').agg({
        '사용금액_백만원': 'sum',
        '수량_천개': 'sum'
    }).reset_index().rename(columns={
        '사용금액_백만원': 'meft_금액_백만원',
        '수량_천개': 'meft_수량_천개'
    })

    display(meft_summary.dtypes)


    # 3. 병합 및 비교
    comp_df = pd.merge(sick_summary, meft_summary, on='진료년월', how='outer')
    comp_df['금액_차이(백만원)'] = comp_df['sick_금액_백만원'] - comp_df['meft_금액_백만원']
    comp_df['수량_차이(천개)'] = comp_df['sick_수량_천개'] - comp_df['meft_수량_천개']
    comp_df = comp_df.sort_values('진료년월').reset_index(drop=True)

    # 4. SICK → MEFT 커버리지 계산
    comp_df['금액_커버리지(%)'] = (
        comp_df['sick_금액_백만원'] / comp_df['meft_금액_백만원'] * 100
    )
    comp_df['수량_커버리지(%)'] = (
        comp_df['sick_수량_천개'] / comp_df['meft_수량_천개'] * 100
    )

    print("\n=== [2015년 월별] SICK → MEFT 커버리지 ===")
    print(comp_df[['진료년월', '금액_커버리지(%)', '수량_커버리지(%)']].to_string(index=False))
    print("\n=== [2015년 전체] sick vs meft 월별 합산 비교 ===")
    display(comp_df)
    # print(comp_df.to_string(index=False))

✅ 발견된 2015년 파일 수: 12개


진료년월               str
sick_금액_백만원    float64
sick_수량_천개     float64
dtype: object

✅ meft_2015 데이터 행 수: 1500


진료년월               str
meft_금액_백만원    float64
meft_수량_천개     float64
dtype: object


=== [2015년 월별] SICK → MEFT 커버리지 ===
  진료년월  금액_커버리지(%)  수량_커버리지(%)
201501   58.778236   41.746374
201502   57.011932   41.430657
201503   57.530398   41.265404
201504   57.631973   41.374098
201505   57.878539   41.755833
201506   57.379353   41.766757
201507   58.604826   43.489793
201508   58.311972   43.122658
201509   57.952849   42.464130
201510   58.068314   42.536206
201511   57.493381   42.414111
201512   56.083849   42.247085

=== [2015년 전체] sick vs meft 월별 합산 비교 ===


,진료년월,sick_금액_백만원,sick_수량_천개,meft_금액_백만원,meft_수량_천개,금액_차이(백만원),수량_차이(천개),금액_커버리지(%),수량_커버리지(%)
0,201501,784940.435487,2336989.553,1335427.0,5598066.0,-550486.564513,-3261076.447,58.778236,41.746374
1,201502,675290.365293,2141140.913,1184472.0,5168011.0,-509181.634707,-3026870.087,57.011932,41.430657
2,201503,789707.117672,2408006.760,1372678.0,5835413.0,-582970.882328,-3427406.240,57.530398,41.265404
3,201504,773578.416615,2337798.334,1342273.0,5650391.0,-568694.583385,-3312592.666,57.631973,41.374098
4,201505,728081.918939,2154307.020,1257948.0,5159296.0,-529866.081061,-3004988.980,57.878539,41.755833
5,201506,696156.724373,1952673.974,1213253.0,4675187.0,-517096.275627,-2722513.026,57.379353,41.766757
6,201507,782505.706165,2106322.856,1335224.0,4843258.0,-552718.293835,-2736935.144,58.604826,43.489793
7,201508,753062.968863,2071536.154,1291438.0,4803823.0,-538375.031137,-2732286.846,58.311972,43.122658
8,201509,750957.647415,2206429.851,1295808.0,5195985.0,-544850.352585,-2989555.149,57.952849,42.464130
9,201510,805716.433572,2344718.577,1387532.0,5512289.0,-581815.566428,-3167570.423,58.068314,42.536206


처방기준 vs 조제기준 sick data 비교



In [18]:
df01 = pd.read_parquet('../data/data_201501.pq')  # 조제
df02 = pd.read_parquet('../sick_처방기준_201501.pq')  # 처방

In [ ]:
df01.sort_values('상병코드').head()

의약품사용금액    784940435487
총사용량         2336989553
dtype: int64


,진료년월,보험구분코드,약효분류번호,약효분류명,상병코드,상병명,의약품사용금액,총사용량
1750,201501,4,112,최면진정제,AA00,콜레라,70581,401
1751,201501,5,112,최면진정제,AA00,콜레라,19180,77
1752,201501,7,112,최면진정제,AA00,콜레라,6090,35
4305,201501,4,113,항전간제,AA00,콜레라,808733,2413
4306,201501,5,113,항전간제,AA00,콜레라,200799,699


In [29]:
df02.sort_values('상병코드').head()

,진료년월,보험구분코드,약효분류번호,약효분류명,상병코드,상병명,의약품사용금액,총사용량
3135,201501,4,112,최면진정제,AA00,콜레라,1106,7
8050,201501,4,114,해열·진통·소염제,AA00,콜레라,4935,89
11068,201501,4,117,정신신경용제,AA00,콜레라,150,12
21798,201501,4,123,자율신경제,AA00,콜레라,4671,27
23997,201501,4,124,진경제,AA00,콜레라,6075,42


In [44]:

print(df01[['의약품사용금액','총사용량']].sum())
print(df02[['의약품사용금액','총사용량']].sum())
meft_df = pd.read_csv('meft_2014-2025.csv')

df00 = meft_df[meft_df['진료년월']=='2015-01'].copy()

print(df00[['사용금액_백만원','수량_천개']].sum())



의약품사용금액    784940435487
총사용량         2336989553
dtype: int64
의약품사용금액    1278030606434
총사용량          4891196777
dtype: int64
사용금액_백만원    1335427.0
수량_천개       5598066.0
dtype: float64


In [50]:
df00[df00.약효분류코드 == 616].head()
df01[df01.약효분류번호 == 616].head()

,진료년월,보험구분코드,약효분류번호,약효분류명,상병코드,상병명,의약품사용금액,총사용량


In [46]:
import glob
import pandas as pd
import numpy as np

# ============================================================
# 설정
# ============================================================

TARGET_MONTH = '201501'

sick_files = glob.glob('../data/data_2015*.pq')

# ------------------------------------------------------------
# 실제 컬럼명 자동 확인
# ------------------------------------------------------------

sample = pd.read_parquet(sick_files[0])

print("SICK 컬럼:")
print(sample.columns.tolist())


# 약효분류코드 후보
eff_candidates = [
    '약효분류코드',
    '약효분류번호',
    '약효분류',
    'meftDivNo',
    '약효분류코드번호'
]

# 상병코드 후보
sickcode_candidates = [
    '상병코드',
    '주상병코드',
    '상병코드1',
    'st3SickSym',
    '질병코드'
]

eff_col = next(
    (c for c in eff_candidates if c in sample.columns),
    None
)

sickcode_col = next(
    (c for c in sickcode_candidates if c in sample.columns),
    None
)

if eff_col is None:
    raise ValueError(
        "❌ 약효분류코드 컬럼을 찾지 못했습니다."
    )

if sickcode_col is None:
    raise ValueError(
        "❌ 상병코드 컬럼을 찾지 못했습니다."
    )

print(f"\n약효분류코드 컬럼: {eff_col}")
print(f"상병코드 컬럼: {sickcode_col}")


# ============================================================
# 1. 처방 SICK 데이터 로드
# ============================================================

sick_list = []

use_cols = [
    '진료년월',
    eff_col,
    sickcode_col,
    '의약품사용금액',
    '총사용량'
]

for file_path in sick_files:

    df = pd.read_parquet(
        file_path,
        columns=use_cols
    )

    # 2015-01만 사용
    df['진료년월'] = (
        df['진료년월']
        .astype(str)
        .str.replace('-', '', regex=False)
    )

    df = df[df['진료년월'] == TARGET_MONTH].copy()

    if len(df) > 0:
        sick_list.append(df)


sick = pd.concat(
    sick_list,
    ignore_index=True
)

print(f"\n처방 SICK 201501 행 수: {len(sick):,}")


# ============================================================
# 2. 약효분류군별 SICK 집계
# ============================================================

sick_eff = (
    sick
    .groupby(eff_col, dropna=False)
    .agg(
        sick_금액=('의약품사용금액', 'sum'),
        sick_수량=('총사용량', 'sum')
    )
    .reset_index()
)


# ============================================================
# 3. MEFT 201501 로드
# ============================================================

meft_df = pd.read_csv('meft_2014-2025.csv')

meft_df['진료년월'] = (
    meft_df['진료년월']
    .astype(str)
    .str.replace('-', '', regex=False)
)

meft = meft_df[
    meft_df['진료년월'] == TARGET_MONTH
].copy()


# MEFT 약효분류군별 집계
meft_eff = (
    meft
    .groupby('약효분류코드', dropna=False)
    .agg(
        meft_금액=('사용금액_백만원', 'sum'),
        meft_수량=('수량_천개', 'sum')
    )
    .reset_index()
)


# ============================================================
# 4. 단위 통일
#    SICK: 원 / 개
#    MEFT: 백만원 / 천개
# ============================================================

sick_eff['sick_금액_백만원'] = (
    sick_eff['sick_금액'] / 1_000_000
)

sick_eff['sick_수량_천개'] = (
    sick_eff['sick_수량'] / 1_000
)


# ============================================================
# 5. 약효분류코드 타입 통일
# ============================================================

sick_eff[eff_col] = (
    sick_eff[eff_col]
    .astype(int).astype(str)
    .str.strip()
)

meft_eff['약효분류코드'] = (
    meft_eff['약효분류코드']
    .astype(int).astype(str)
    .str.strip()
)


# ============================================================
# 확인 1
# MEFT에는 있지만 처방 SICK에는 없는 약효분류군
# ============================================================

coverage = pd.merge(
    meft_eff,
    sick_eff,
    left_on='약효분류코드',
    right_on=eff_col,
    how='left',
    indicator=True
)

meft_only = coverage[
    coverage['_merge'] == 'left_only'
].copy()

print("\n" + "=" * 70)
print("① MEFT에는 있지만 처방 SICK에는 없는 약효분류군")
print("=" * 70)

print(
    f"MEFT 약효분류군 수: {len(meft_eff):,}"
)

print(
    f"SICK에 없는 약효분류군 수: {len(meft_only):,}"
)

if len(meft_only) > 0:

    meft_only['MEFT_금액_백만원'] = (
        meft_only['meft_금액']
    )

    meft_only['MEFT_수량_천개'] = (
        meft_only['meft_수량']
    )

    display(
        meft_only[
            [
                '약효분류코드',
                'MEFT_금액_백만원',
                'MEFT_수량_천개'
            ]
        ]
        .sort_values(
            'MEFT_금액_백만원',
            ascending=False
        )
    )


# ============================================================
# 확인 2
# 약효분류군별 금액/수량 커버리지
# ============================================================

coverage['금액_커버리지(%)'] = (
    coverage['sick_금액_백만원']
    / coverage['meft_금액']
    * 100
)

coverage['수량_커버리지(%)'] = (
    coverage['sick_수량_천개']
    / coverage['meft_수량']
    * 100
)

coverage['금액_차이_백만원'] = (
    coverage['meft_금액']
    - coverage['sick_금액_백만원'].fillna(0)
)

coverage['수량_차이_천개'] = (
    coverage['meft_수량']
    - coverage['sick_수량_천개'].fillna(0)
)

print("\n" + "=" * 70)
print("② 약효분류군별 MEFT 대비 처방 SICK 커버리지")
print("=" * 70)

display(
    coverage[
        [
            '약효분류코드',
            'meft_금액',
            'sick_금액_백만원',
            '금액_커버리지(%)',
            'meft_수량',
            'sick_수량_천개',
            '수량_커버리지(%)',
            '금액_차이_백만원',
            '수량_차이_천개'
        ]
    ]
    .sort_values(
        '금액_커버리지(%)',
        na_position='first'
    )
)


# ============================================================
# 커버리지 요약
# ============================================================

print("\n--- 커버리지 요약 ---")

valid_amount = coverage[
    coverage['meft_금액'] > 0
]['금액_커버리지(%)']

valid_qty = coverage[
    coverage['meft_수량'] > 0
]['수량_커버리지(%)']

print(
    f"금액 커버리지 중앙값: "
    f"{valid_amount.median():.2f}%"
)

print(
    f"수량 커버리지 중앙값: "
    f"{valid_qty.median():.2f}%"
)


# ============================================================
# 확인 3
# 약효분류코드는 있지만 상병코드가 없는 SICK 데이터
# ============================================================

# NaN / 빈 문자열 / 'nan' 등을 상병코드 없음으로 처리
sickcode_str = (
    sick[sickcode_col]
    .astype('string')
    .str.strip()
)

no_sickcode = (
    sickcode_str.isna()
    | sickcode_str.eq('')
    | sickcode_str.eq('nan')
    | sickcode_str.eq('None')
)

# 약효분류코드가 있는 행만
has_eff = (
    sick[eff_col].notna()
    & sick[eff_col].astype(str).str.strip().ne('')
)

no_sickcode_df = sick[
    no_sickcode & has_eff
].copy()


no_sickcode_summary = pd.DataFrame({
    '행수': [len(no_sickcode_df)],
    '금액_백만원': [
        no_sickcode_df['의약품사용금액'].sum() / 1_000_000
    ],
    '수량_천개': [
        no_sickcode_df['총사용량'].sum() / 1_000
    ]
})

print("\n" + "=" * 70)
print("③ 약효분류코드는 있지만 상병코드가 없는 처방 SICK")
print("=" * 70)

display(no_sickcode_summary)


# 약효분류군별로도 확인
no_sickcode_by_eff = (
    no_sickcode_df
    .groupby(eff_col, dropna=False)
    .agg(
        행수=('의약품사용금액', 'size'),
        금액=('의약품사용금액', 'sum'),
        수량=('총사용량', 'sum')
    )
    .reset_index()
)

no_sickcode_by_eff['금액_백만원'] = (
    no_sickcode_by_eff['금액'] / 1_000_000
)

no_sickcode_by_eff['수량_천개'] = (
    no_sickcode_by_eff['수량'] / 1_000
)

print("\n--- 상병코드 없는 처방의 약효분류군별 규모 ---")

display(
    no_sickcode_by_eff[
        [
            eff_col,
            '행수',
            '금액_백만원',
            '수량_천개'
        ]
    ]
    .sort_values(
        '금액_백만원',
        ascending=False
    )
)

SICK 컬럼:
['진료년월', '보험구분코드', '약효분류번호', '약효분류명', '상병코드', '상병명', '의약품사용금액', '총사용량']

약효분류코드 컬럼: 약효분류번호
상병코드 컬럼: 상병코드

처방 SICK 201501 행 수: 165,524

① MEFT에는 있지만 처방 SICK에는 없는 약효분류군
MEFT 약효분류군 수: 125
SICK에 없는 약효분류군 수: 32


,약효분류코드,MEFT_금액_백만원,MEFT_수량_천개
30,223,4222.0,510134.0
95,616,2263.0,19.0
17,210,501.0,594.0
32,231,247.0,25789.0
27,220,161.0,4.0
104,632,152.0,5.0
61,312,120.0,4604.0
70,326,99.0,111.0
118,729,92.0,5840.0
57,266,83.0,94.0



② 약효분류군별 MEFT 대비 처방 SICK 커버리지


,약효분류코드,meft_금액,sick_금액_백만원,금액_커버리지(%),meft_수량,sick_수량_천개,수량_커버리지(%),금액_차이_백만원,수량_차이_천개
5,116,33.0,NaN,NaN,1098.0,NaN,NaN,33.000000,1098.000
17,210,501.0,NaN,NaN,594.0,NaN,NaN,501.000000,594.000
23,216,58.0,NaN,NaN,430.0,NaN,NaN,58.000000,430.000
27,220,161.0,NaN,NaN,4.0,NaN,NaN,161.000000,4.000
30,223,4222.0,NaN,NaN,510134.0,NaN,NaN,4222.000000,510134.000
...,...,...,...,...,...,...,...,...,...
0,111,3183.0,3048.730412,95.781665,7242.0,6933.200,95.735985,134.269588,308.800
48,252,60.0,57.594417,95.990695,321.0,260.988,81.304673,2.405583,60.012
89,490,1153.0,1112.385699,96.477511,1260.0,1215.027,96.430714,40.614301,44.973
116,722,149.0,145.227982,97.468444,16.0,15.628,97.675000,3.772018,0.372



--- 커버리지 요약 ---
금액 커버리지 중앙값: 63.41%
수량 커버리지 중앙값: 59.71%

③ 약효분류코드는 있지만 상병코드가 없는 처방 SICK


,행수,금액_백만원,수량_천개
0,0,0.0,0.0



--- 상병코드 없는 처방의 약효분류군별 규모 ---


,약효분류번호,행수,금액_백만원,수량_천개
